# 🔬 VeriSight MCP — Empirical Forensic Benchmark Suite
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adityarajtiwari880/verisight/blob/main/benchmarks/verisight_forensic_benchmark.ipynb)
[![PyPI version](https://img.shields.io/pypi/v/verisight.svg?color=blue)](https://pypi.org/project/verisight/)

Empirical forensic validation of **VeriSight v2.2.5** against real-world camera captures (COCO val2017 & OpenCV) and real-world claim fraud assets.

Evaluates **In-Memory Streaming (0 Local Disk Footprint)** across:
1. **Authentic Camera Captures**: Real-world Flickr camera photos from COCO val2017 & OpenCV organic captures.
2. **Real-World Fraud Claims**: Generative AI inpainting, fabricated collision bumper dents, altered tax invoices, altered voucher codes, punctured delivery parcels, broken screens, and curdled mold spoilage.


In [ ]:
# Step 1: Install VeriSight and Benchmark Dependencies in Colab
!pip install -q --upgrade --no-cache-dir verisight==2.2.5 opencv-python-headless matplotlib seaborn scikit-learn pandas tqdm
print('VeriSight v2.2.5 and benchmark dependencies installed successfully!')


In [ ]:
# Step 2: Import Forensic Agent & Configure SSL
import os, sys, time, urllib.request, ssl, json
ssl._create_default_https_context = ssl._create_unverified_context
import cv2, numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

# Import forensic agent from verisight (with auto-reload for active kernels)
import importlib
try:
    import verisight
    import verisight.backend
    import verisight.backend.agent
    import verisight.backend.forensics
    importlib.reload(verisight)
    importlib.reload(verisight.backend)
    importlib.reload(verisight.backend.forensics)
    importlib.reload(verisight.backend.agent)
    from verisight.backend.agent import forensic_agent
    from verisight.backend.aws_adapter import aws_adapter
    v_ver = getattr(verisight, '__version__', 'unknown')
except Exception:
    try:
        from verisight.backend.agent import forensic_agent
        from verisight.backend.aws_adapter import aws_adapter
        v_ver = '2.2.5'
    except ImportError:
        import backend as verisight
        v_ver = getattr(verisight, '__version__', 'local')
        from backend.agent import forensic_agent
        from backend.aws_adapter import aws_adapter

print(f'VeriSight Forensic Agent v{v_ver} loaded successfully!')


In [ ]:
# Step 3: Run the Empirical Forensic Benchmark Suite (Streaming In-Memory)
# Evaluates against organic Flickr/OpenCV camera captures & real-world fraud assets.

authentic_records = []
tampered_records = []

# 1. Authentic COCO real-world Flickr captures
coco_ids = [139, 285, 632, 724, 776, 785, 802, 872, 885, 1000, 1268, 1296, 1353, 1490, 1503]
for cid in coco_ids:
    authentic_records.append({
        'id': f'coco_{cid}',
        'url': f'http://images.cocodataset.org/val2017/{cid:012d}.jpg',
        'ground_truth': 'AUTHENTIC',
        'category': 'Natural Camera Scene (COCO Flickr)',
        'claim_type': 'Commercial Delivery Package Claim'
    })

# 2. Authentic OpenCV real camera captures
opencv_samples = [
    ('opencv_fruits', 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/fruits.jpg', 'Organic Fresh Produce'),
    ('opencv_messi', 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/messi5.jpg', 'Sports Motion Action'),
    ('opencv_butterfly', 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/butterfly.jpg', 'Wildlife Macro'),
    ('opencv_baboon', 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/baboon.jpg', 'Animal Fur Texture'),
    ('opencv_building', 'https://raw.githubusercontent.com/opencv/opencv/master/samples/data/building.jpg', 'Architectural Facade')
]
for name, url, cat in opencv_samples:
    authentic_records.append({
        'id': name,
        'url': url,
        'ground_truth': 'AUTHENTIC',
        'category': cat,
        'claim_type': 'Commercial Delivery Package Claim'
    })

# 3. Ground-truth real-world fraudulent claims hosted on GitHub raw
repo_raw = 'https://raw.githubusercontent.com/Adityarajtiwari880/verisight/main/backend'
repo_fraud_samples = [
    ('gt_strawberries_inpainted', f'{repo_raw}/sample_real_strawberries_inpainted.jpg', 'AI Inpainted Mold Fraud', 'Commercial Delivery Package Claim'),
    ('gt_car_bumper_dent', f'{repo_raw}/sample_real_car_bumper_dent.jpg', 'AI Car Bumper Dent Fraud', 'Automotive Collision Claim'),
    ('gt_invoice_tampered', f'{repo_raw}/sample_real_invoice_tampered.jpg', 'Tampered Tax Invoice Price', 'Commercial Invoice Claim'),
    ('gt_voucher_tampered', f'{repo_raw}/sample_real_voucher_tampered.jpg', 'Altered Voucher Promo Code', 'Commercial Voucher Claim'),
    ('gt_ai_diffusion_fraud', f'{repo_raw}/2_ai_inpainted_fraud.jpg', 'Generative AI Diffusion Inpaint', 'Commercial Delivery Package Claim'),
    ('gt_delivery_box_puncture', f'{repo_raw}/sample_real_delivery_box_puncture.jpg', 'Inpainted Parcel Crush Damage', 'Commercial Delivery Package Claim'),
    ('gt_cracked_windshield', f'{repo_raw}/sample_real_cracked_windshield.jpg', 'Inpainted Auto Glass Bullseye', 'Automotive Collision Claim'),
    ('gt_spoiled_milk_mold', f'{repo_raw}/sample_real_spoiled_milk_mold.jpg', 'Inpainted Dairy Food Spoilage', 'Commercial Delivery Package Claim'),
    ('gt_altered_prescription', f'{repo_raw}/sample_real_altered_prescription.jpg', 'Altered Medical Rx Dosage', 'Commercial Invoice Claim'),
    ('gt_broken_screen_phone', f'{repo_raw}/sample_real_broken_screen_phone.jpg', 'Inpainted Electronics Screen Crack', 'Commercial Delivery Package Claim')
]
for name, url, cat, ctype in repo_fraud_samples:
    tampered_records.append({
        'id': name,
        'url': url,
        'ground_truth': 'TAMPERED',
        'category': cat,
        'claim_type': ctype
    })

# Interleave 2:1 (Authentic:Tampered) for an empirical distribution
test_records = []
a_idx, t_idx = 0, 0
while a_idx < len(authentic_records) or t_idx < len(tampered_records):
    if a_idx < len(authentic_records):
        test_records.append(authentic_records[a_idx])
        a_idx += 1
    if a_idx < len(authentic_records):
        test_records.append(authentic_records[a_idx])
        a_idx += 1
    if t_idx < len(tampered_records):
        test_records.append(tampered_records[t_idx])
        t_idx += 1

print(f'Executing empirical forensic benchmark on {len(test_records)} ground-truth samples...')
print(f'Authentic Camera Photos: {len(authentic_records)} | Real Fraud Claims: {len(tampered_records)}')

y_true, y_pred_prob, y_pred_binary = [], [], []
latencies = []
tp, tn, fp, fn = 0, 0, 0, 0

print(f'\n{"Sample ID":<32} | {"Ground Truth":<12} | {"Verdict":<26} | {"Risk":<6} | {"Status"}')
print('-' * 95)

for rec in test_records:
    aws_adapter.clear_test_ledger()
    try:
        req = urllib.request.Request(rec['url'], headers={'User-Agent': 'Mozilla/5.0'})
        with urllib.request.urlopen(req, timeout=10) as resp:
            raw_b = resp.read()
        img = cv2.imdecode(np.frombuffer(raw_b, np.uint8), cv2.IMREAD_COLOR)
        if img is None: continue

        t0 = time.perf_counter()
        res = forensic_agent.run_investigation(img, raw_bytes=raw_b, claim_type=rec['claim_type'], claim_id=f"BM-{rec['id']}")
        lat = (time.perf_counter() - t0) * 1000.0
        latencies.append(lat)

        v = res['verdict']
        risk = res['tamper_probability']
        is_pred_tamper = (v in ['REJECT_SYNTHETIC_TAMPER', 'HUMAN_IN_THE_LOOP_REVIEW', 'REJECT_DUPLICATE_CLAIM_FRAUD']) or (risk >= 45.0)

        gt_b = 1 if rec['ground_truth'] == 'TAMPERED' else 0
        pr_b = 1 if is_pred_tamper else 0
        y_true.append(gt_b)
        y_pred_prob.append(risk / 100.0)
        y_pred_binary.append(pr_b)

        if gt_b == 1 and pr_b == 1:
            tp += 1
            status = 'CORRECT (Caught Fraud)'
        elif gt_b == 0 and pr_b == 0:
            tn += 1
            status = 'CORRECT (Auto-Approved)'
        elif gt_b == 0 and pr_b == 1:
            fp += 1
            status = 'FALSE POSITIVE'
        elif gt_b == 1 and pr_b == 0:
            fn += 1
            status = 'FALSE NEGATIVE'

        print(f"{rec['id']:<32} | {rec['ground_truth']:<12} | {v:<26} | {risk:>5.1f}% | {status}")
    except Exception as e:
        print(f"Error processing {rec['id']}: {e}")
        continue

print(f'\nCompleted Evaluation: {len(y_true)} samples | TP={tp}, TN={tn}, FP={fp}, FN={fn}')


In [ ]:
# Step 4: Statistical Metrics, Confusion Matrix & ROC Curve
total = len(y_true)
acc = (tp + tn) / total * 100.0
spec = tn / max(1, (tn + fp)) * 100.0
rec = tp / max(1, (tp + fn)) * 100.0
prec = tp / max(1, (tp + fp)) * 100.0
f1 = 2 * prec * rec / max(1e-5, (prec + rec))
mean_lat = np.mean(latencies) if latencies else 0.0

print('=' * 60)
print('  VERISIGHT MCP — EMPIRICAL FORENSIC BENCHMARK RESULTS')
print('=' * 60)
print(f' Overall Accuracy         : {acc:.1f}%')
print(f' Specificity (Authentic)  : {spec:.1f}%  ({tn} True Negative, {fp} False Positive)')
print(f' Recall (Fraud Caught)    : {rec:.1f}%  ({tp} True Positive, {fn} False Negative)')
print(f' Precision                : {prec:.1f}%')
print(f' F1-Score                 : {f1:.1f}%')
print(f' Mean Latency             : {mean_lat:.2f} ms')
print('=' * 60)

# Render Confusion Matrix and ROC Curve
cm = confusion_matrix(y_true, y_pred_binary)
fpr, tpr, _ = roc_curve(y_true, y_pred_prob)
roc_auc = auc(fpr, tpr)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax1,
            xticklabels=['Pred Authentic', 'Pred Fraud'],
            yticklabels=['Actual Authentic', 'Actual Fraud'])
ax1.set_title('VeriSight Forensic Confusion Matrix', fontsize=14, fontweight='bold')

ax2.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.3f})')
ax2.plot([0, 1], [0, 1], color='navy', lw=1.5, linestyle='--')
ax2.set_xlim([0.0, 1.0])
ax2.set_ylim([0.0, 1.05])
ax2.set_xlabel('False Positive Rate', fontsize=11)
ax2.set_ylabel('True Positive Rate', fontsize=11)
ax2.set_title('Receiver Operating Characteristic (ROC)', fontsize=14, fontweight='bold')
ax2.legend(loc='lower right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
